In [1]:
import requests, time, re
import pandas as pd

In [2]:
def get_indie_ids(limit=280):
    url = "https://steamspy.com/api.php?request=tag&tag=Indie"
    resp = requests.get(url, timeout=30)
    data = resp.json()

    def parse_owners(s):
        nums = re.findall(r'\d+', str(s).replace(',', ''))
        return (int(nums[0]) + int(nums[1])) // 2 if len(nums) >= 2 else 0

    games = [(int(k), parse_owners(v.get('owners', '0'))) for k, v in data.items()]
    games.sort(key=lambda x: x[1], reverse=True)
    return [g[0] for g in games[:limit]]

indie_ids = get_indie_ids(280)
print(f"Получено {len(indie_ids)} ID инди-игр из SteamSpy")

Получено 280 ID инди-игр из SteamSpy


In [3]:
def scrape_steam_game(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us&l=english"
    try:
        r = requests.get(url, timeout=12)
        raw = r.json()
        if not raw.get(str(appid), {}).get('success'):
            return None
        d = raw[str(appid)]['data']
        if d.get('type') != 'game':
            return None

        po = d.get('price_overview') or {}
        plat = d.get('platforms') or {}
        cats = [c['id'] for c in (d.get('categories') or [])]

        yr_match = re.search(r'\b(20\d{2}|19\d{2})\b',
                             d.get('release_date', {}).get('date', ''))

        return {
            'app_id':             appid,
            'name':               d.get('name', ''),
            'developer':          (d.get('developers') or [''])[0],
            'publisher':          (d.get('publishers') or [''])[0],
            'release_year':       int(yr_match.group()) if yr_match else None,
            'is_free':            int(d.get('is_free', False)),
            'price_usd':          po.get('final', 0) / 100,
            'original_price_usd': po.get('initial', 0) / 100,
            'discount_percent':   po.get('discount_percent', 0),
            'required_age':       int(d.get('required_age', 0) or 0),
            'metacritic_score':   (d.get('metacritic') or {}).get('score', 0),
            'recommendations':    (d.get('recommendations') or {}).get('total', 0),
            'achievements_count': (d.get('achievements') or {}).get('total', 0),
            'dlc_count':          len(d.get('dlc') or []),
            'platforms_count':    sum([plat.get('windows', False),
                                       plat.get('mac', False),
                                       plat.get('linux', False)]),
            'has_sin gle_player':  int(2  in cats),
            'has_multiplayer':    int(1  in cats),
            'has_co_op':          int(9  in cats),
            'genres':             ', '.join(g['description']
                                            for g in (d.get('genres') or [])),
        }
    except Exception:
        return None

In [4]:
games_data, failed = [], 0

for i, appid in enumerate(indie_ids):
    result = scrape_steam_game(appid)
    if result:
        games_data.append(result)
    else:
        failed += 1

    if (i + 1) % 30 == 0:
        print(f"Обработано: {i+1}/{len(indie_ids)} | "
              f"Собрано: {len(games_data)} | Ошибок: {failed}")
    time.sleep(1.6)   # соблюдаем rate-limit Steam: 200 req / 5 min

print(f"\nИтого: собрано данных по {len(games_data)} играм, пропущено {failed}")

Обработано: 30/280 | Собрано: 30 | Ошибок: 0
Обработано: 60/280 | Собрано: 60 | Ошибок: 0
Обработано: 90/280 | Собрано: 90 | Ошибок: 0
Обработано: 120/280 | Собрано: 120 | Ошибок: 0
Обработано: 150/280 | Собрано: 150 | Ошибок: 0
Обработано: 180/280 | Собрано: 180 | Ошибок: 0
Обработано: 210/280 | Собрано: 210 | Ошибок: 0
Обработано: 240/280 | Собрано: 240 | Ошибок: 0
Обработано: 270/280 | Собрано: 270 | Ошибок: 0

Итого: собрано данных по 280 играм, пропущено 0


In [5]:
df_raw = pd.DataFrame(games_data)
df_raw.to_excel("../data/steam_indie_games.xlsx", index=False)
print(f"Файл сохранён: {df_raw.shape[0]} строк × {df_raw.shape[1]} столбцов")
df_raw.head(3)

Файл сохранён: 280 строк × 19 столбцов


,app_id,name,developer,publisher,release_year,is_free,price_usd,original_price_usd,discount_percent,required_age,metacritic_score,recommendations,achievements_count,dlc_count,platforms_count,has_single_player,has_multiplayer,has_co_op,genres
0,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",2017,1,0.00,0.00,0,13,0,1756391,37,0,1,0,1,0,"Action, Adventure, Massively Multiplayer, Free..."
1,1623730,Palworld,Pocketpair,Pocketpair,2024,0,22.49,29.99,25,0,0,349359,57,1,1,1,1,1,"Action, Adventure, Indie, RPG, Early Access"
2,304930,Unturned,Smartly Dressed Games,Smartly Dressed Games,2017,1,0.00,0.00,0,0,0,7531,88,1,3,1,1,1,"Action, Adventure, Casual, Indie, Free To Play"
